# Stochastic Optimization

## 2.1 Genetic Algorithms

<a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" align="left" src="https://i.creativecommons.org/l/by-sa/4.0/80x15.png" /></a>&nbsp;| Dennis G. Wilson | <a href="https://d9w.github.io/evolution/">https://d9w.github.io/evolution/</a>

## Outline

1. [Problème](#problem)
2. [Population](#population)
3. [Evaluation](#evaluation)
4. [Sélection](#selection)
5. [Crossover](#crossover)
6. [Mutation](#mutation)

 • Souvent utilisé pour l’optimisation **binaire** ou **discrète**  
 • Adapté aux problèmes contraints  

<img src="https://github.com/d9w/evolution/raw/master/imgs/ga.png">

## <a id="problem"></a>Problème

Pour comprendre l'algorithme génétique, nous allons l'utiliser pour résoudre le problème du **voyageur de commerce (TSP)**. 
  
Pour rappel, dans le TSP, un$\cdot$e commercial$\cdot$e doit visiter un ensemble de villes d'une liste donnée exactement une fois. L'objectif est de **minimiser la distance totale parcourue**.  
  
Ce problème trouve de nombreuses applications, notamment dans la gestion de la chaîne d'approvisionnement, la production manufacturière et la navigation.

Le TSP est un problème **NP-difficile** qui a fait l'objet de nombreuses études.  
  
Sa complexité maximale peut être évaluée en considérant une recherche exhaustive de toutes les solutions possibles: pour **n** villes, elle est de l'ordre **O(n!)**.  
  
Diverses méthodes ont permis de réduire cette complexité de calcul, notamment grâce à des algorithmes récents conçus pour les ordinateurs quantiques.  
  
Les algorithmes génétiques présentent de multiples avantages pour ce type de problème, tels que la **capacité à traiter un grand nombre de villes**, la **flexibilité** face aux modifications (comme l'ajout de contraintes) et la **facilité de parallélisation**. Toutefois, ils ne garantissent pas l'obtention d'une solution exacte, mais seulement d'une **approximation**.

Pour créer notre problème de TSP, nous générerons aléatoirement des villes sur un repère en deux dimensions et utiliserons leur distance euclidienne comme coût de déplacement entre elles.

In [ ]:
!pip install seaborn

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import math
import seaborn as sns

np.random.seed(1234)
rng = np.random.default_rng()
sns.set_theme(style='ticks')

In [ ]:
n_cities = 10
cities = np.random.rand(n_cities, 2)

In [ ]:
plt.scatter(cities[:, 0], cities[:, 1])
plt.title("Cities");

Pour notre problème à 10 villes, combien y a t'il de chemins possibles?  

Qu'en est-il d'un problème à 20 villes?  

A 30 villes?

In [ ]:
n = n_cities
print(f"O(n!): {math.factorial(n)}")

 Faisons l'hypothèse optimiste que notre machine met $10^{-12}$s pour évaluer 1 chemin possible  
  
| Nombre de villes |    Temps de calcul (s) |
|:---------------- | ----------------------:|
| 10               |    $3.63\cdot 10^{-6}$ |
| 15               |                 $1.31$ |
| 20               |              $2432902$ |
| 30               |    $2.65\cdot 10^{20}$ |


 
 • $2432902$s ≈ 0.77 ans  

 • $2.65\cdot 10^{20}$s ≈ $84111130077$ millénaires soit 500 fois l’âge de l’Univers

## <a id="population"></a>Population

On constitue la population initiale: on génère des solutions aléatoire, qu'on va appeler **individus**

On a plusieurs choix pour représenter les individus. Dans notre exemple, une solution sera représentée par une liste des villes dans l'ordre de visite. Cette représentation sera appelée **génome**.

Par la suite, les individus seront modifiés par des opérations de combinaison et de mutation de leur génome. 

Chaque solution doit inclure chaque ville *exactement une fois*, ce qui correspond à une **permutation des indices de la liste des villes**. Il s'agit d'une manière concise de représenter le trajet, qui se prête également bien aux mutations, comme nous le verrons plus loin.

In [ ]:
order0 = rng.permutation(n_cities)
order1 = rng.permutation(n_cities)
print(order0)
print(order1)

In [ ]:
fig = plt.figure(figsize=(14, 4))
ax = plt.subplot(1, 2, 1)
ind = order0
plt.title("Order 0")
ax.scatter(cities[ind, 0], cities[ind, 1])
ax.plot(cities[ind, 0], cities[ind, 1], 'k')
for i in range(n_cities):
    ax.annotate(order0[i], (cities[ind[i], 0]+0.01, cities[ind[i], 1]+0.01), c='g')
ax = plt.subplot(1, 2, 2)
ind = order1
plt.title("Order 1")
ax.scatter(cities[ind, 0], cities[ind, 1])
ax.plot(cities[ind, 0], cities[ind, 1], 'k')
for i in range(n_cities):
    ax.annotate(order1[i], (cities[ind[i], 0]+0.01, cities[ind[i], 1]+0.01), c='g')

Pour la population initiale de notre algorithme génétique, nous allons créer 100 permutations aléatoires.  

Vu qu'il existe $10!$ solutions possibles pour le problème du voyageur de commerce (TSP) asymétrique à 10 villes, 100 représente un nombre très faible par rapport à l'espace des solutions total.   

Les algorithmes génétiques ont toutefois tendance à utiliser des populations assez vastes afin de représenter des solutions diversifiées.

In [ ]:
n_population = 100
population = np.array([rng.permutation(n_cities) for i in range(n_population)])
population[:5]

## Evaluation

L'étape d'évaluation d'un algorithme génétique est identique à celle des autres algorithmes stochastiques: il faut calculer la valeur de la fonction objectif pour chacun des individus générés.  
Ici, la fonction objectif représente la distance totale parcourue en suivant l'ordre défini par la solution.

In [ ]:
d = np.zeros((n_cities, n_cities))
for i in range(n_cities):
    for j in range(i):
        d[i, j] = np.sqrt((cities[i, 0] - cities[j, 0])**2 + (cities[i,1] - cities[j, 1])**2)
        d[j, i] = d[i, j]
d[:3, :3]

In [ ]:
def total_distance(order, distances):
    t = 0
    for i in range(1, len(order)):
        t += distances[order[i-1], order[i]]
    return t

In [ ]:
print("order 0:", total_distance(order0, d))
print("order 1:", total_distance(order1, d))

In [ ]:
def evaluate(population, distances):
    fitness = np.zeros(len(population))
    for i in range(len(population)):
        fitness[i] = total_distance(population[i], distances)
    return fitness

In [ ]:
fitness = evaluate(population, d)
for i in range(3):
    print(population[i], fitness[i])
print("Minimum: ")
print(population[np.argmin(fitness)], np.min(fitness), np.max(fitness))

## <a id="selection"></a>Sélection

Quels individus devraient transmettre leur information génétique à la génération suivante ?  
  
**Première approche**: un mécanisme simple consistant à sélectionner un pourcentage des meilleurs individus, par exemple les 20 % les plus performants.  
  
C'est ce que l'on appelle la **sélection par troncature**.  
  
Une telle méthode pourrait fonctionner pour des problèmes simples, **exempts de minima locaux**. Toutefois, **nous perdrions une diversité génétique importante**, qui constitue pourtant l'un des principaux avantages des grandes populations dans les algorithmes génétiques.

In [ ]:
def truncation_selection(population, fitness, p=0.2):
    n_elites = int(np.floor(len(population) * p))
    elites = np.argsort(fitness)[:n_elites]
    return population[elites], fitness[elites]

In [ ]:
elites, efit = truncation_selection(population, fitness)
plt.scatter(fitness, np.zeros(len(fitness)), color='k', label='all')
plt.scatter(efit, np.ones(len(efit)), color='b', label='truncation')
plt.legend();

Afin de mieux préserver la diversité génétique, nous allons permettre à tout membre de la population d'avoir une chance d'être sélectionné, proportionnelement à la valeur de son fitness.   

C'est ce que l'on appelle la sélection **proportionnelle au fitness**.

In [ ]:
def fp_selection(population, fitness):
    p = (np.max(fitness) - fitness)
    p /= np.sum(p)
    rng = np.random.default_rng()
    ind = rng.choice(len(population), p=p)
    return population[ind], fitness[ind]

In [ ]:
fp_fits = np.zeros(len(efit))
for i in range(len(efit)):
    p, f = fp_selection(population, fitness)
    fp_fits[i] = f

In [ ]:
plt.scatter(fitness, np.zeros(len(fitness)), color='k', label='all')
plt.scatter(efit, np.ones(len(efit)), color='b', label='truncation')
plt.scatter(fp_fits, 2*np.ones(len(fp_fits)), color='r', label='FP')
plt.legend();

L'une des propriétés avantageuses de certaines stratégies d'évolution est l'**invariance** par rapport à certaines transformations de l'espace de recherche.  
  
Pour obtenir cette invariance, on peut utiliser une séléction qui se base sur l'**ordre** des fitness des individus et non pas leurs valeurs extactes.  

Cela rend la recherche plus généralisable aux espaces qui ne sont pas convexes ou qui sont mal conditionnés (par exemple, ceux présentant un relief accidenté).

Hansen, Nikolaus, et al. "Impacts of invariance in search: When CMA-ES and PSO face ill-conditioned and non-separable problems." Applied Soft Computing 11.8 (2011): 5755-5769.

Dans un algorithme génétique, on peut obtenir la même invariance en effectuant une **sélection par tournoi**.  
  
On sélectionne aléatoirement un sous-ensemble d'individus de notre population. On leur fait faire un "tournoi" et on  retient le meilleur individu dans notre population suivante.   
  
Comme cette méthode ne dépend pas de la valeur du fitness, mais plutôt du classement des individus au sein du tournoi, elle est également invariante par rapport aux transformations qui préservent l'ordre.

<div class="alert alert-success">
    <h3>Exercice 1</h3>  
    
Complétez la définition de la sélection par tournoi suivante, qui utilise une taille de tournoi `t_size=3`. Représentez graphiquement les individus sélectionnés et comparez les résultats avec ceux de la sélection par troncature et de la sélection proportionnelle au fitness.
</div>

In [ ]:
def tournament_selection(population, fitness, t_size=3):
    rng = np.random.default_rng()
    tournament = rng.choice(len(population), )
    ind = 
    return population[ind], fitness[ind]

In [ ]:
t_fits = np.zeros(len(efit))
for i in range(len(efit)):
    p, f = tournament_selection(population, fitness)
    t_fits[i] = f

In [ ]:
plt.scatter(fitness, np.zeros(len(fitness)), color='k', label='all')
plt.scatter(efit, np.ones(len(efit)), color='b', label='truncation')
plt.scatter(fp_fits, 2*np.ones(len(fp_fits)), color='r', label='FP')
plt.scatter(t_fits, 3*np.ones(len(t_fits)), color='g', label='tournament')
plt.legend();

## <a id="crossover"></a>Crossover

Compte tenu de la taille importante de notre population, existe-t-il un moyen de **combiner de bons individus** afin d'aboutir à de meilleures solutions?  
  
Par exemple, pourrions-nous créer un individu héritant d'informations provenant de deux individus parents?  
  
C'est le principe du croisement (ou *crossover*). Il s'inspire de la reproduction sexuée, où le patrimoine génétique de deux individus parents est combiné pour engendrer une descendance.  
  
Examinons les différentes manières de combiner deux individus.

<img src="https://github.com/d9w/evolution/raw/master/imgs/crossover.png" width="80%" height="auto">

In [ ]:
def one_point(p1, p2):
    rng = np.random.default_rng()
    x = rng.choice(np.arange(1, np.minimum(len(p1)-1, len(p2)-1)))
    return np.concatenate((p1[:x], p2[x:])), np.concatenate((p2[:x],p1[x:]))

In [ ]:
parent1, _ = tournament_selection(population, fitness)
parent2, _ = tournament_selection(population, fitness)
print(parent1, parent2)
child1, child2 = one_point(parent1, parent2)
print("crossover: ")
print(child1, child2)

La méthode de croisement à un point est courante. Inspirée de la biologie, elle s'avère utile pour combiner des solutions dans le cadre de divers problèmes. Lorsque l'on porte le nombre de points d'échange au-delà de un, on parle de croisement à *k* points, une technique utile lorsque les dimensions du problème sont indépendantes.

Dans le problème du voyageur de commerce, toutefois, les dimensions du problème ne sont pas indépendantes; il existe une contrainte imposant de visiter toutes les villes sans qu'aucune ne soit visitée deux fois.  
  
Nous utiliserons plutôt un opérateur de croisement qui respecte ces contraintes. Il s'agit de l'opérateur de recombinaison d'arêtes (edge recombination operator), applicable à tout problème de recombinaison sur un graphe orienté.

<img src="https://upload.wikimedia.org/wikipedia/commons/e/e5/Genetic_ero_crossover.svg" alt="Genetic ero crossover.svg" width="50%" height="auto">

https://en.wikipedia.org/wiki/Edge_recombination_operator
    
Whitley, L. Darrell, Timothy Starkweather, and D'Ann Fuquay. "Scheduling problems and traveling salesmen: The genetic edge recombination operator." ICGA. Vol. 89. 1989. [pdf](TSP_crossover.pdf)

Cet opérateur effectue une sélection aléatoire au sein du voisinage de chaque nœud. Comme sa programmation est assez longue, nous utiliserons la [définition](https://github.com/anyoptimization/pymoo/blob/master/pymoo/operators/crossover/erx.py) issue de la bibliothèque [pymoo](https://pymoo.org/index.html).

In [ ]:
!pip install pymoo

In [ ]:
from pymoo.operators.crossover import erx
parent1 = population[0]
parent2 = population[1]
print("=== PARENTS ===")
print("Parent 1 :", parent1)
print("Parent 2 :", parent2)
print("\n=== ADJACENCY MATRIX 1 ===")
adj1 = erx.calc_adjency_matrix(parent1)
for node, neighbours in adj1.items():
    print(f"{int(node)} : {sorted(int(x) for x in neighbours)}")
print("\n=== ADJACENCY MATRIX 2 ===")
adj2 = erx.calc_adjency_matrix(parent2)
for node, neighbours in adj2.items():
    print(f"{int(node)} : {sorted(int(x) for x in neighbours)}")
child = erx.erx(parent1, parent2)
print("\n=== CHILD ===")
print([int(x) for x in child])

## <a id="mutation"></a>Mutation

Le croisement combine des informations génétiques déjà présentes au sein de la population. Pour explorer des configurations sortant du cadre des combinaisons existantes, il est nécessaire de modifier directement les individus.  
  
Cette opération s'effectue par le biais de la **mutation**, appliquée soit directement aux individus issus de la sélection, soit, plus fréquemment, aux nouveaux individus générés par le croisement.  
  
En règle générale, les opérateurs de mutation ne doivent pas altérer radicalement l'individu; ils ne modifient qu'une faible proportion de ses gènes. La mutation uniforme en est un exemple : elle consiste à rééchantillonner aléatoirement certains gènes. Un taux de mutation couramment utilisé dans ce cas est `1/n_genes`, ce qui signifie qu'en moyenne, un seul gène est modifié.

Tout comme pour le croisement, nous devons créer un opérateur de mutation qui respecte les contraintes du problème du voyageur de commerce. Un tel opérateur consisterait à inverser l'ordre d'une unique paire de villes choisie au hasard.

<div class="alert alert-success">
    <h3>Exercice 2</h3>

Complétez la définition suivante de l'opérateur de mutation, qui inverse une seule paire aléatoire de villes.
</div>

In [ ]:
def mutate(ind):
    return ind

In [ ]:
print(population[0], population[1])
child = erx.erx(population[0], population[1])
print(np.array(child))
mutate(child)

## <a id="ga"></a>The Genetic Algorithm

Maintenant que nous disposons de tous les éléments, nous pouvons les assembler pour former l'algorithme génétique complet.  
  
Nous utiliserons deux méthodes de sélection distinctes : tout d'abord, une sélection par troncature des meilleurs individus, qui seront directement transférés à la génération suivante. Cette technique, appelée **élitisme**, permet de préserver les meilleures solutions d'une génération à l'autre.  
  
Ensuite, nous recourrons à la sélection par tournoi pour choisir les parents destinés au croisement.  
  
Enfin, nous appliquerons une mutation au résultat du croisement et intégrerons ce nouvel individu dans la population suivante.

In [ ]:
def ga_step(population):
    fitness = evaluate(population, d)
    next_pop, _ = truncation_selection(population, fitness)
    while len(next_pop) < len(population):
        parent1, _ = tournament_selection(population, fitness)
        parent2, _ = tournament_selection(population, fitness)
        child = erx.erx(parent1, parent2)
        child = mutate(child)
        next_pop = np.concatenate((next_pop, [child]))
    return next_pop, fitness

In [ ]:
n_cities = 20
cities = np.random.rand(n_cities, 2)
d = np.zeros((n_cities, n_cities))
for i in range(n_cities):
    for j in range(i):
        d[i, j] = np.sqrt((cities[i, 0] - cities[j, 0])**2 + (cities[i,1] - cities[j, 1])**2)
        d[j, i] = d[i, j]

In [ ]:
n_population = 100
n_gen = 100
population = np.array([rng.permutation(n_cities) for i in range(n_population)])
minfit = np.zeros(n_gen)
for i in range(n_gen):
    population, fitness = ga_step(population)
    minfit[i] = np.min(fitness)
    if i > 2 and minfit[i] < minfit[i-1]:
        print(i, minfit[i])

In [ ]:
plt.plot(minfit);

<div class="alert alert-success">
Étudiez deux hyperparamètres de l'algorithme génétique : la taille de la population et le nombre d'élites. Quel est leur impact sur la recherche? Les élites sont-elles nécessaires ?
</div>

<div class="alert alert-info">
    <h3>Discussion</h3>

Combien d'évaluations ont été réalisées dans le cadre de cet l'algorithme génétique? Cela correspond-il au nombre total de permutations explorées? En ordre de grandeur, combien de permutations ont été calculées par rapport au nombre total de permutations de villes possibles? L'algorithme génétique a-t-il convergé vers la meilleure solution possible? À quel moment l'algorithme génétique doit-il s'arrêter ?
</div>